# Contaminants

## Imports

In [1]:
import pandas as pd
import pprint as pp
import json

# https://www.datacamp.com/tutorial/settingwithcopywarning-pandas
# https://saturncloud.io/blog/how-to-disable-warnings-in-jupyter-notebook/
import warnings

# Not good, but the warnings were annoying me!!
# warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore')

## Load original contaminants dataset

In [2]:
raw_contaminants_csv_file_path = './../data/raw/contaminants/contaminants.csv'

In [3]:
df = pd.read_csv(raw_contaminants_csv_file_path)

## Original contaminants dataset EDA

In [4]:
# 21 Rows and 3 Columns
df.shape

(21, 3)

In [5]:
# Display the first 5 rows of the DataFrame
df.head()

,Codi_Contaminant,Desc_Contaminant,Unitats
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³


In [6]:
# Display the last 5 rows of the DataFrame
df.tail()

,Codi_Contaminant,Desc_Contaminant,Unitats
16,114,O3*,µg/m³
17,996,Flow 2 (Mesura interna equip),NaN
18,997,Flow 1 (Mesura interna equip),NaN
19,998,Flow C (Mesura interna equip),
20,999,Biomassa Black Carbon,%


In [7]:
# Summary of the DataFrame including column names, their data types, and their non-null values count
df.info()

# 'Codi_Contaminant' -> int code
# 'Desc_Contaminant' -> object/string (text)
# 'Unitats'          -> object/string (text)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Codi_Contaminant  21 non-null     int64 
 1   Desc_Contaminant  21 non-null     object
 2   Unitats           19 non-null     object
dtypes: int64(1), object(2)
memory usage: 632.0+ bytes


## Duplicated data and missing values

In [8]:
# Missing values for each column
missing_values = df.isnull().sum()
missing_values

# 2 (3 in reality) NULL or NaN values in the 'Unitats' column
# Rows 17, 18, 19

Codi_Contaminant    0
Desc_Contaminant    0
Unitats             2
dtype: int64

In [9]:
# Missing values for each column
missing_values = df.isna().sum()
missing_values

# 2 (3 in reality) NULL or NaN values in the 'Unitats' column
# Rows 17, 18, 19

Codi_Contaminant    0
Desc_Contaminant    0
Unitats             2
dtype: int64

In [10]:
# Get duplicated rows
duplicates = df.duplicated().sum()
duplicates

# No duplicated rows
# In reality we have multiple contaminants with 2 ids (same 'Desc_Contaminant' but with an extra '*' char)

0

In [11]:
# Show all data
df

,Codi_Contaminant,Desc_Contaminant,Unitats
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³
5,10,PM10,µg/m³
6,12,NOx,µg/m³
7,14,O3,µg/m³
8,22,Black Carbon,µg/m³
9,101,SO2*,µg/m³


## Changing columns names

In [12]:
# Original DataFrame columns names
print(list(df.columns))

['Codi_Contaminant', 'Desc_Contaminant', 'Unitats']


In [13]:
# New DataFrame columns names
new_df_columns_names = ['code', 'description', 'unit']

# Change columns of DataFrame
df.columns = new_df_columns_names

In [14]:
# Show all data
df

,code,description,unit
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³
5,10,PM10,µg/m³
6,12,NOx,µg/m³
7,14,O3,µg/m³
8,22,Black Carbon,µg/m³
9,101,SO2*,µg/m³


## Remove contaminants 

In [15]:
# Remove contaminants that "don't matter"
contaminants_to_remove_from_df = [22, 996, 997, 998, 999]

# df is now the old df minus the contaminants which the code are in 'contaminants_to_remove_from_df' list
df = df[~df.code.isin(contaminants_to_remove_from_df)]

# Show all data
df

,code,description,unit
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³
5,10,PM10,µg/m³
6,12,NOx,µg/m³
7,14,O3,µg/m³
9,101,SO2*,µg/m³
10,106,CO*,mg/m³


## EDA conclusions

- All contaminants have the __same unit (with the exception of the contaminant 'CO' / 'CO*' using mg/m³)__.
    - __All others__ are measured in __µg/m³__.
- There is a relationship between the __previous contaminant code__ and the __new contaminant code (plus 100)__.
- There is a relationship between the __previous contaminant description__ and the __new contaminant description (+ '*' char)__.
- There are 16 entries (contaminants). But since they are 'duplicated', __there are only 8 "valid" contaminants__.
    - __We will only study/forecast only the 3 most important to the city of barcelona__.

## Standardizing text columns

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16 entries, 0 to 16
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   code         16 non-null     int64 
 1   description  16 non-null     object
 2   unit         16 non-null     object
dtypes: int64(1), object(2)
memory usage: 512.0+ bytes


In [20]:
# Standardizing text data

df.unit = df.unit.str.strip()
# df.unit = df.unit.astype('category')

df.description = df.description.str.strip().str.upper()
# df.description = df.description.astype('category')

In [21]:
# Show all data
df

,code,description,unit
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³
5,10,PM10,µg/m³
6,12,NOX,µg/m³
7,14,O3,µg/m³
9,101,SO2*,µg/m³
10,106,CO*,mg/m³


In [22]:
# Remove '*' from the new contaminants description
df.description = df.description.str.replace('*', '')

# Replace 'NOX' for 'NOx'
df.description = df.description.str.replace('X', 'x')

# Possible problem with data onward 
# PM2.5 -> The '.' can be a problem and will be replaced with '_'
df.description = df.description.str.replace('.', '_')

# Show all data
df

,code,description,unit
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2_5,µg/m³
5,10,PM10,µg/m³
6,12,NOx,µg/m³
7,14,O3,µg/m³
9,101,SO2,µg/m³
10,106,CO,mg/m³


## Save data to csv file

In [23]:
processed_contaminants_csv_file_path = './../data/processed/contaminants/contaminants.csv'

In [24]:
processed_contaminants_df = df.copy()
processed_contaminants_df.to_csv(processed_contaminants_csv_file_path, index=False)

In [25]:
test_df = pd.read_csv(processed_contaminants_csv_file_path)
test_df

,code,description,unit
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2_5,µg/m³
5,10,PM10,µg/m³
6,12,NOx,µg/m³
7,14,O3,µg/m³
8,101,SO2,µg/m³
9,106,CO,mg/m³


## Create and test code to implement ContaminantManager class

In [26]:
# Each contaminant has 2 codes
df['description'].value_counts()

SO2      2
CO       2
NO       2
NO2      2
PM2_5    2
PM10     2
NOx      2
O3       2
Name: description, dtype: int64

In [27]:
# Get distinct contaminant names
distinct_contaminant_names = set(df.description)

# Dictionary to hold contaminant data without duplications
# and with the multiple contaminants codes in a list 
contaminant_data_dict = {}

In [28]:
for contaminant_name in distinct_contaminant_names:
    
    # Creates a DataFrame with 2 rows (each contaminants have two codes - two rows)
    unique_contaminant_df = df[df.description == contaminant_name]
    
    # Iterate the DataFrame rows
    for index, row in unique_contaminant_df.iterrows():
        
        # Is already a key in the dictionary? No, then is added to it
        if row.description not in contaminant_data_dict:
            
            contaminant_data_dict[row.description] = {}
            contaminant_data_dict[row.description]['unit'] = row.unit
            contaminant_data_dict[row.description]['codes'] = [] # list object
        
        # Add the code to the list of codes
        contaminant_data_dict[row.description]['codes'].append(row.code)

In [29]:
contaminant_data_dict

{'PM10': {'unit': 'µg/m³', 'codes': [10, 110]},
 'NOx': {'unit': 'µg/m³', 'codes': [12, 112]},
 'PM2_5': {'unit': 'µg/m³', 'codes': [9, 109]},
 'NO': {'unit': 'µg/m³', 'codes': [7, 107]},
 'SO2': {'unit': 'µg/m³', 'codes': [1, 101]},
 'NO2': {'unit': 'µg/m³', 'codes': [8, 108]},
 'O3': {'unit': 'µg/m³', 'codes': [14, 114]},
 'CO': {'unit': 'mg/m³', 'codes': [6, 106]}}

In [30]:
contaminant_data_dict['PM10']

{'unit': 'µg/m³', 'codes': [10, 110]}

In [31]:
contaminant_data_dict['CO']

{'unit': 'mg/m³', 'codes': [6, 106]}

In [32]:
contaminant_data_dict['PM2_5']

{'unit': 'µg/m³', 'codes': [9, 109]}

## Save contaminants in JSON format

* https://www.geeksforgeeks.org/python/python-difference-between-json-dump-and-json-dumps/
* https://www.datacamp.com/tutorial/json-data-python

In [33]:
contaminants_json_file_path = './../data/processed/contaminants/contaminants.json'

In [34]:
with open(contaminants_json_file_path, 'w') as json_file:
    json.dump(contaminant_data_dict, json_file)

In [35]:
json_data_test_read = {}

with open(contaminants_json_file_path) as json_file:
    json_data_test_read = json.load(json_file)

In [36]:
json_data_test_read

{'PM10': {'unit': 'µg/m³', 'codes': [10, 110]},
 'NOx': {'unit': 'µg/m³', 'codes': [12, 112]},
 'PM2_5': {'unit': 'µg/m³', 'codes': [9, 109]},
 'NO': {'unit': 'µg/m³', 'codes': [7, 107]},
 'SO2': {'unit': 'µg/m³', 'codes': [1, 101]},
 'NO2': {'unit': 'µg/m³', 'codes': [8, 108]},
 'O3': {'unit': 'µg/m³', 'codes': [14, 114]},
 'CO': {'unit': 'mg/m³', 'codes': [6, 106]}}

## Test ContaminantManagerJSON class

In [78]:
# https://www.geeksforgeeks.org/python/python-import-module-from-different-directory/
import sys

# Adding code/generic_code to the system path
sys.path.insert(0, './generic_code')

In [83]:
from ContaminantManagerJSON import ContaminantManagerJSON

In [84]:
c_manager_obj = ContaminantManagerJSON(contaminants_json_file_path)

In [93]:
print("Get all contaminants [ContaminantManagerJSON class]")
pp.pprint(c_manager_obj.get_all_contaminants())

Get all contaminants [ContaminantManagerJSON class]
{'CO': {'codes': [6, 106], 'unit': 'mg/m³'},
 'NO': {'codes': [7, 107], 'unit': 'µg/m³'},
 'NO2': {'codes': [8, 108], 'unit': 'µg/m³'},
 'NOx': {'codes': [12, 112], 'unit': 'µg/m³'},
 'O3': {'codes': [14, 114], 'unit': 'µg/m³'},
 'PM10': {'codes': [10, 110], 'unit': 'µg/m³'},
 'PM2_5': {'codes': [9, 109], 'unit': 'µg/m³'},
 'SO2': {'codes': [1, 101], 'unit': 'µg/m³'}}


In [96]:
print("Get contaminant data by code [ContaminantManagerJSON class]\n")

template = "'{0}' data => '{1}'"
print(template.format("PM10", c_manager_obj.get_contaminant_data_by_code("PM10")))
print(template.format("PM2_5", c_manager_obj.get_contaminant_data_by_code("PM2_5")))
print(template.format("SO2", c_manager_obj.get_contaminant_data_by_code("SO2")))
print(template.format("CO", c_manager_obj.get_contaminant_data_by_code("CO")))

Get contaminant data by code [ContaminantManagerJSON class]
'PM10' data => '{'unit': 'µg/m³', 'codes': [10, 110], 'description': 'PM10'}'
'PM2_5' data => '{'unit': 'µg/m³', 'codes': [9, 109], 'description': 'PM2_5'}'
'SO2' data => '{'unit': 'µg/m³', 'codes': [1, 101], 'description': 'SO2'}'
'CO' data => '{'unit': 'mg/m³', 'codes': [6, 106], 'description': 'CO'}'
'PM10' data -> {'unit': 'µg/m³', 'codes': [10, 110], 'description': 'PM10'}
'PM2_5' data -> {'unit': 'µg/m³', 'codes': [9, 109], 'description': 'PM2_5'}
'SO2' data -> {'unit': 'µg/m³', 'codes': [1, 101], 'description': 'SO2'}
'CO' data -> {'unit': 'mg/m³', 'codes': [6, 106], 'description': 'CO'}


In [98]:
print("Get contaminant units by code [ContaminantManagerJSON class]\n")

template = "'{0}' units => '{1}'"
print(template.format("PM10", c_manager_obj.get_units_by_code("PM10")))
print(template.format("PM2_5", c_manager_obj.get_units_by_code("PM2_5")))
print(template.format("SO2", c_manager_obj.get_units_by_code("SO2")))
print(template.format("CO", c_manager_obj.get_units_by_code("CO")))

Get contaminant units by code [ContaminantManagerJSON class]

'PM10' units => 'µg/m³'
'PM2_5' units => 'µg/m³'
'SO2' units => 'µg/m³'
'CO' units => 'mg/m³'


In [103]:
print("Get contaminant description by code [ContaminantManagerJSON class]\n")

template = "Code '{0}' ('{1}') => '{2}'"

print(template.format("10", "PM10", c_manager_obj.get_description_by_code(10)))
print(template.format("110", "PM10", c_manager_obj.get_description_by_code(110)))

print(template.format("9", "PM2_5", c_manager_obj.get_description_by_code(9)))
print(template.format("109", "PM2_5", c_manager_obj.get_description_by_code(109)))

print(template.format("1", "SO2", c_manager_obj.get_description_by_code(1)))
print(template.format("101", "SO2", c_manager_obj.get_description_by_code(101)))

print(template.format("6", "CO", c_manager_obj.get_description_by_code(6)))
print(template.format("106", "CO", c_manager_obj.get_description_by_code(106)))

Get contaminant description by code [ContaminantManagerJSON class]

Code '10' ('PM10') => 'PM10'
Code '110' ('PM10') => 'PM10'
Code '9' ('PM2_5') => 'PM2_5'
Code '109' ('PM2_5') => 'PM2_5'
Code '1' ('SO2') => 'SO2'
Code '101' ('SO2') => 'SO2'
Code '6' ('CO') => 'CO'
Code '106' ('CO') => 'CO'


In [ ]:
# Acabar comentários de JSOJ c_manager
# Refazer c_manager original (tem duplicados e tem de ser transformado internamente em dict)
# Acabar listagem de ficheiros de data

In [26]:
import contaminants as c_manager

In [27]:
c_manager_obj = c_manager.ContaminantManager(original_contaminants_path)

In [28]:
c_manager_obj.DEFAULT_COLUMNS

['code', 'description', 'units']

In [29]:
c_manager_obj.get_column_names()

['code', 'description', 'units']

In [30]:
c_manager_obj.get_all_contaminants()

,code,description,units
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³
5,10,PM10,µg/m³
6,12,NOx,µg/m³
7,14,O3,µg/m³
8,22,Black Carbon,µg/m³
9,101,SO2*,µg/m³


In [31]:
[c_manager_obj.get_description_by_code(1),
c_manager_obj.get_description_by_code(101)]

['SO2', 'SO2*']

In [32]:
[c_manager_obj.get_units_by_code(1),
c_manager_obj.get_units_by_code(101)]

['µg/m³', 'µg/m³']

In [33]:
[c_manager_obj.get_contaminant_data_by_code(1),
c_manager_obj.get_contaminant_data_by_code(101)]

[{'code': 1, 'description': 'SO2', 'units': 'µg/m³'},
 {'code': 101, 'description': 'SO2*', 'units': 'µg/m³'}]

In [34]:
from pathlib import Path

In [35]:
root_dir = './../data/raw/'
root = Path(root_dir)

In [36]:
data_raw_paths = dict()

In [37]:
for file in root.rglob("*.csv"):
    # Get parts of the path relative to the root
    parts = file.relative_to(root).parts
    if len(parts) == 3:
        if parts[0] not in data_raw_paths:
            data_raw_paths[parts[0]] = dict()
        if parts[1] not in data_raw_paths[parts[0]].keys():
            data_raw_paths[parts[0]][parts[1]] = dict()
        #print(parts)
        data_raw_paths[parts[0]][parts[1]][parts[2][:-3]] = {}
        
    else:
        data_raw_paths[parts[0]] = str(file)

In [38]:
for file in root.rglob("*.csv"):
    # Get parts of the path relative to the root
    parts = file.relative_to(root).parts
    print(parts)

('contaminants', 'contaminants.csv')
('measurements', '2018', '2018_06.csv')
('measurements', '2018', '2018_07.csv')
('measurements', '2018', '2018_08.csv')
('measurements', '2018', '2018_09.csv')
('measurements', '2018', '2018_10.csv')
('measurements', '2018', '2018_11.csv')
('measurements', '2018', '2018_12.csv')
('measurements', '2019', '2019_01.csv')
('measurements', '2019', '2019_02.csv')
('measurements', '2019', '2019_03.csv')
('measurements', '2019', '2019_04.csv')
('measurements', '2019', '2019_05.csv')
('measurements', '2019', '2019_06.csv')
('measurements', '2019', '2019_07.csv')
('measurements', '2019', '2019_08.csv')
('measurements', '2019', '2019_09.csv')
('measurements', '2019', '2019_10.csv')
('measurements', '2019', '2019_11.csv')
('measurements', '2019', '2019_12.csv')
('measurements', '2020', '2020_01.csv')
('measurements', '2020', '2020_02.csv')
('measurements', '2020', '2020_03.csv')
('measurements', '2020', '2020_04.csv')
('measurements', '2020', '2020_05.csv')
('m

In [40]:
if not parts[0] in data_raw_paths.keys():
            data_raw_paths[parts[0]] = dict()
        if not parts[1] in data_raw_paths[parts[0]].keys():
            data_raw_paths[parts[0]][parts[1]] = dict()
        #print(parts)
        data_raw_paths[parts[0]][parts[1]][parts[2][:-3]] = {}

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 3)